In [8]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from pyspark.sql import functions as F

print("🔄 CFO Summary (Final Fixed Version)...")

df_gold = spark.read.format("delta").load("Tables/dbo/gold_ai_production")
df_gold.show(3, truncate=False)

# YOUR columns exactly
df = df_gold.select(
    F.col("site"), F.col("department"), 
    F.col("monthly_budget_eur").cast("double").alias("planned"),
    F.col("monthly_actual_eur").cast("double").alias("actual"),
    F.col("variance_pct").cast("double").alias("var_pct"),
    F.col("cfo_alert"), F.col("priority")
)

# Metrics
summary = df.agg(
    F.sum("actual").alias("total_actual"),
    F.sum("planned").alias("total_planned"),
    F.avg("var_pct").alias("avg_var"),
    F.sum(F.when(F.col("priority") == "HIGH", 1).otherwise(0)).alias("high_alerts"),
    F.count("*").alias("records")
).collect()[0]

total_actual = float(summary["total_actual"])
total_planned = float(summary["total_planned"])
avg_var = float(summary["avg_var"])
high_alerts = int(summary["high_alerts"])
records = int(summary["records"])

def fmt_eur(x): return f"€{abs(x):,.0f}"

variance_desc = "OVER" if avg_var > 0 else "UNDER"
suggestion = (
    "Reallocate €" + fmt_eur((total_planned-total_actual)*0.3) + " from low-priority to HIGH variance" 
    if high_alerts > 0 and total_actual > total_planned else "Stable - continue monitoring"
)

summary_text = (
    f"ERP: {records} records ({avg_var:.1f}% avg {variance_desc} budget). "
    f"€{fmt_eur(total_actual)} actual vs €{fmt_eur(total_planned)} planned. "
    f"{high_alerts} HIGH alerts. Next: €{fmt_eur(total_actual*1.08)}. {suggestion}."
)

print("📊 SUMMARY:", summary_text)

# High priority (FIXED column name)
high_priority = df.filter(F.col("priority") == "HIGH").select("department", "cfo_alert", "var_pct").collect()

html_body = f"""
<html><body style="font-family:Arial;max-width:600px">
  <h1 style="color:#1a73e8">Storm AI ERP Summary</h1>
  <p style="font-size:16px">{summary_text}</p>
  
  <div style="background:#f8f9fa;padding:20px;border-radius:8px">
    <h3>HIGH Alerts ({high_alerts}):</h3>
"""
if high_priority:
    for r in high_priority[:3]:
        html_body += f'<p>• {r["department"]}: {r["cfo_alert"]} ({r["var_pct"]:.1f}%)</p>'
else:
    html_body += '<p>✅ All stable</p>'

html_body += """
  </div>
  
  <div style="text-align:center;margin:24px 0">
    <a href="https://app.fabric.microsoft.com" 
       style="background:#1a73e8;color:white;padding:16px 32px;font-size:18px;
              text-decoration:none;border-radius:8px;font-weight:600">
      📊 Dashboard → Full ML Analysis
    </a>
  </div>
  
  <p style="color:#666;font-size:12px">MSc AI + Storm Tech | Auto-generated</p>
</body></html>
"""

# SEND
sender_email = "smrutiakshay2805@gmail.com"
sender_password = "rpelnoqlzyqgezgg"
recipient_emails = ["smrutipote0502@gmail.com", "smruti.po1106@gmail.com"]

msg = MIMEMultipart()
msg['From'] = sender_email
msg['To'] = ", ".join(recipient_emails)
msg['Subject'] = f"Storm AI: {high_alerts} HIGH – {avg_var:.0f}% variance"

msg.attach(MIMEText(html_body, 'html'))

try:
    server = smtplib.SMTP('smtp.gmail.com', 587)
    server.starttls()
    server.login(sender_email, sender_password)
    server.send_message(msg)
    server.quit()
    print(f"✅ EMAILS SENT! ({high_alerts} HIGH)")
except Exception as e:
    print(f"❌ Error: {e}")

# ✅ FIXED final display (uses var_pct)
print("\nHigh priority:")
df.filter(F.col("priority") == "HIGH").select("department", "cfo_alert", "var_pct").show()




StatementMeta(, 9390732e-e34c-4056-a4c5-57609b0d8c92, 10, Finished, Available, Finished)

🔄 CFO Summary (Final Fixed Version)...
+-------------------+--------+-------------+--------------------+------------------+------------+-------------+-----------+--------------+----------------+---------+--------+-------------------+
|date               |site    |department   |monthly_budget_eur  |monthly_actual_eur|variance_pct|budget_status|iso_anomaly|zscore_anomaly|business_anomaly|cfo_alert|priority|pred_error_pct     |
+-------------------+--------+-------------+--------------------+------------------+------------+-------------+-----------+--------------+----------------+---------+--------+-------------------+
|2025-04-30 00:00:00|Limerick|International|1.0608469477528282E7|8921764.110238165 |-15.9       |Under        |1          |0             |0               |OK       |LOW     |0.37876868742968367|
|2025-05-31 00:00:00|Limerick|International|9754307.33135218    |8852983.599868406 |-9.24       |Under        |1          |0             |0               |OK       |LOW     |2.81192